# ROAD Hyperparameter Search For Cheap-IG

Этот notebook запускает staged-search гиперпараметров `cheap-IG` на `ROAD MoRF` и затем сравнивает `IG`, `NAA` и finalists на отдельном holdout.

In [1]:
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

from modules import IG
from modules.baseline_utils import baseline_display_name
from modules.road_benchmark import benchmark_classifier_road, classifier_method_spec
from modules.road_hparam_search import (
    default_cheap_ig_search_space,
    render_road_search_report,
    run_staged_road_search,
    split_search_holdout,
)


## Параметры

In [2]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 100

CLASSIFIER_LAYER = "model.6"
SEARCH_FRACTION = 0.7
SPLIT_SEED = 0

ROAD_NOISE = 0.01
ROAD_NOISE_SEED = 0
CLEAR_EVERY = 8
FD_EPS = 1e-3

CACHE_ROOT = Path("output/road_cache")
OUTPUT_DIR = Path("output/road_hparam_search_oxford_pets_100_pred_top1")

REFRESH_CORE = False
REFRESH_METHODS = False
REFRESH_EVALUATIONS = False


In [3]:
def collect_oxford_pets_images(image_dir=OXFORD_PETS_DIR, n_images=N_IMAGES):
    image_dir = Path(image_dir)
    if not image_dir.exists():
        raise FileNotFoundError(f"Directory not found: {image_dir}")
    image_paths = sorted(
        [
            path
            for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
        ],
        key=lambda path: path.name.lower(),
    )
    if len(image_paths) < n_images:
        raise RuntimeError(f"Expected at least {n_images} images in {image_dir}, found {len(image_paths)}")
    return image_paths[:n_images]


image_paths = collect_oxford_pets_images()
search_space = default_cheap_ig_search_space()
split = split_search_holdout(image_paths, search_fraction=SEARCH_FRACTION, seed=SPLIT_SEED)

len(image_paths), len(split["search_image_paths"]), len(split["holdout_image_paths"])


(100, 70, 30)

In [4]:
def segment_sample_count(segment_end, n_steps, segment_start=0.0):
    count = 0
    for step_idx in range(1, int(n_steps) + 1):
        alpha = step_idx / float(n_steps)
        if segment_end < 1.0:
            if alpha >= segment_start and alpha < segment_end:
                count += 1
        else:
            if alpha >= segment_start and alpha <= segment_end:
                count += 1
    return count


search_space

{
    f"segment_end={segment_end:g}": {
        f"n_steps={n_steps}": segment_sample_count(segment_end, n_steps)
        for n_steps in search_space["n_steps_values"]
    }
    for segment_end in search_space["segment_end_values"]
}


{'segment_end=0.1': {'n_steps=24': 2,
  'n_steps=48': 4,
  'n_steps=96': 9,
  'n_steps=192': 19},
 'segment_end=0.12': {'n_steps=24': 2,
  'n_steps=48': 5,
  'n_steps=96': 11,
  'n_steps=192': 23},
 'segment_end=0.15': {'n_steps=24': 3,
  'n_steps=48': 7,
  'n_steps=96': 14,
  'n_steps=192': 28},
 'segment_end=0.2': {'n_steps=24': 4,
  'n_steps=48': 9,
  'n_steps=96': 19,
  'n_steps=192': 38}}

In [ ]:
def compute_dataset_mean_rgb(image_paths):
    rgb_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0
    for path in image_paths:
        _, image_np = IG.load_image(str(path))
        rgb_sum += image_np.sum(axis=(0, 1))
        pixel_count += image_np.shape[0] * image_np.shape[1]
    if pixel_count == 0:
        raise RuntimeError("image_paths is empty")
    return tuple((rgb_sum / pixel_count).tolist())


baseline_mean_rgb = compute_dataset_mean_rgb(image_paths)
baseline_mean_rgb


In [5]:
results = run_staged_road_search(
    image_paths=[str(path) for path in image_paths],
    layer_name=CLASSIFIER_LAYER,
    search_space=search_space,
    search_fraction=SEARCH_FRACTION,
    seed=SPLIT_SEED,
    cache_root=CACHE_ROOT,
    noise=ROAD_NOISE,
    noise_seed=ROAD_NOISE_SEED,
    fd_eps=FD_EPS,
    clear_every=CLEAR_EVERY,
    refresh_core=REFRESH_CORE,
    refresh_methods=REFRESH_METHODS,
    refresh_evaluations=REFRESH_EVALUATIONS,
    verbose=False,
)

artifacts = render_road_search_report(results, output_dir=OUTPUT_DIR)
artifacts["finalists"]


[{'label': 'best_quality',
  'method_name': 'cheap-ig[0,0.2]/positive/k32000',
  'config': {'selection_mode': 'positive',
   'selection_top_k': 32000,
   'segment_start': 0.0,
   'segment_end': 0.2,
   'fill_mode': 'zero',
   'fill_rho': None},
  'config_key': 'cheapig_91e99f5d',
  'config_step_key': 'cheapig_91e99f5d_n24',
  'n_steps': 24,
  'segment_step_count': 4,
  'aoc_mean': 12.086630159238029,
  'runtime_mean': 0.6439260529574572,
  'fill_mode': 'zero',
  'fill_rho': None,
  'segment_end': 0.2,
  'selection_top_k': 32000},
 {'label': 'fastest_pareto',
  'method_name': 'cheap-ig[0,0.1]/positive/k4000/fill-naa_scaled-rho1.2',
  'config': {'selection_mode': 'positive',
   'selection_top_k': 4000,
   'segment_start': 0.0,
   'segment_end': 0.1,
   'fill_mode': 'naa_scaled',
   'fill_rho': 1.2},
  'config_key': 'cheapig_a6aa6a5d',
  'config_step_key': 'cheapig_a6aa6a5d_n24',
  'n_steps': 24,
  'segment_step_count': 2,
  'aoc_mean': 11.837107505400974,
  'runtime_mean': 0.602909171443

In [6]:
display(Markdown(Path(artifacts["search_report_path"]).read_text(encoding="utf-8")))
display(Markdown(Path(artifacts["holdout_report_path"]).read_text(encoding="utf-8")))


# ROAD Hyperparameter Search

- layer_name=`model.6`
- n_images_total=100
- search_images=70
- holdout_images=30
- split_seed=0
- cache_root=`output/road_cache`

## Finalists

| Label | Method | AOC | Runtime (s) | top_k | segment_end | fill | seg steps | n_steps |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| best_quality | cheap-ig[0,0.2]/positive/k32000 | 12.0866 | 0.6439 | 32000 | 0.2 | zero | 4 | 24 |
| fastest_pareto | cheap-ig[0,0.1]/positive/k4000/fill-naa_scaled-rho1.2 | 11.8371 | 0.6029 | 4000 | 0.1 | hybrid rho=1.2 | 2 | 24 |
| best_balanced | cheap-ig[0,0.15]/positive/k4000/fill-naa_scaled-rho0.6 | 12.0480 | 0.6244 | 4000 | 0.15 | hybrid rho=0.6 | 3 | 24 |

## Stage A

| Method | AOC | Runtime (s) | Benchmark (s) | delta vs IG | delta vs NAA | seg steps | Status | Reason |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| k=32000, seg=0.2, zero, n=48 | 12.7864 | 1.1147 | 2.2719 | — | — | 9 | selected | pareto;best_segment_0.2 |
| k=32000, seg=0.2, hybrid rho=0.8, n=48 | 12.7400 | 1.1160 | 2.2731 | — | — | 9 | pruned | — |
| k=16000, seg=0.2, hybrid rho=0.8, n=48 | 12.7060 | 1.1246 | 2.2890 | — | — | 9 | pruned | — |
| k=16000, seg=0.2, zero, n=48 | 12.6933 | 1.1259 | 2.2937 | — | — | 9 | pruned | — |
| k=4000, seg=0.15, zero, n=48 | 12.6733 | 1.0540 | 2.1962 | — | — | 7 | selected | pareto;best_segment_0.15;fastest_within_1pct |
| k=10000, seg=0.2, hybrid rho=0.8, n=48 | 12.6577 | 1.1248 | 2.3029 | — | — | 9 | pruned | — |
| k=10000, seg=0.15, zero, n=48 | 12.6357 | 1.0645 | 2.2171 | — | — | 7 | pruned | — |
| k=4000, seg=0.2, zero, n=48 | 12.6131 | 1.1146 | 2.2751 | — | — | 9 | pruned | — |
| k=4000, seg=0.15, hybrid rho=0.8, n=48 | 12.6112 | 1.0494 | 2.2038 | — | — | 7 | selected | pareto |
| k=32000, seg=0.15, hybrid rho=0.8, n=48 | 12.6015 | 1.0717 | 2.2218 | — | — | 7 | pruned | — |
| k=16000, seg=0.15, zero, n=48 | 12.5992 | 1.0902 | 2.2413 | — | — | 7 | pruned | — |
| k=16000, seg=0.15, hybrid rho=0.8, n=48 | 12.5964 | 1.0701 | 2.2265 | — | — | 7 | pruned | — |
| k=10000, seg=0.15, hybrid rho=0.8, n=48 | 12.5910 | 1.0890 | 2.2562 | — | — | 7 | pruned | — |
| k=4000, seg=0.2, hybrid rho=0.8, n=48 | 12.5848 | 1.1210 | 2.3089 | — | — | 9 | pruned | — |
| k=10000, seg=0.2, zero, n=48 | 12.5825 | 1.1294 | 2.2983 | — | — | 9 | pruned | — |
| k=4000, seg=0.12, zero, n=48 | 12.5728 | 1.0636 | 2.2009 | — | — | 5 | pruned | best_segment_0.12 |
| k=32000, seg=0.15, zero, n=48 | 12.5726 | 1.0685 | 2.2139 | — | — | 7 | pruned | — |
| k=4000, seg=0.12, hybrid rho=0.8, n=48 | 12.5673 | 1.0789 | 2.2357 | — | — | 5 | pruned | — |
| k=6000, seg=0.2, hybrid rho=0.8, n=48 | 12.5259 | 1.1292 | 2.3096 | — | — | 9 | pruned | — |
| k=6000, seg=0.15, zero, n=48 | 12.4973 | 1.0497 | 2.2027 | — | — | 7 | pruned | — |
| k=6000, seg=0.2, zero, n=48 | 12.4948 | 1.1158 | 2.2893 | — | — | 9 | pruned | — |
| k=8000, seg=0.2, zero, n=48 | 12.4736 | 1.1240 | 2.3065 | — | — | 9 | pruned | — |
| k=8000, seg=0.2, hybrid rho=0.8, n=48 | 12.4726 | 1.1344 | 2.3091 | — | — | 9 | pruned | — |
| k=6000, seg=0.15, hybrid rho=0.8, n=48 | 12.4603 | 1.0564 | 2.2119 | — | — | 7 | pruned | — |
| k=32000, seg=0.12, zero, n=48 | 12.4535 | 1.0096 | 2.1277 | — | — | 5 | selected | pareto |
| k=6000, seg=0.12, hybrid rho=0.8, n=48 | 12.4467 | 1.0139 | 2.1671 | — | — | 5 | pruned | — |
| k=10000, seg=0.12, zero, n=48 | 12.4464 | 1.0029 | 2.1183 | — | — | 5 | selected | pareto |
| k=8000, seg=0.15, hybrid rho=0.8, n=48 | 12.4386 | 1.0666 | 2.2201 | — | — | 7 | pruned | — |
| k=8000, seg=0.15, zero, n=48 | 12.4369 | 1.0621 | 2.2144 | — | — | 7 | pruned | — |
| k=32000, seg=0.12, hybrid rho=0.8, n=48 | 12.4356 | 1.0171 | 2.1277 | — | — | 5 | pruned | — |
| k=10000, seg=0.12, hybrid rho=0.8, n=48 | 12.4302 | 1.0062 | 2.1332 | — | — | 5 | pruned | — |
| k=4000, seg=0.1, hybrid rho=0.8, n=48 | 12.4223 | 1.0324 | 2.1643 | — | — | 4 | selected | best_segment_0.1 |
| k=16000, seg=0.12, zero, n=48 | 12.3756 | 1.0179 | 2.1429 | — | — | 5 | pruned | — |
| k=4000, seg=0.1, zero, n=48 | 12.3654 | 1.0496 | 2.1757 | — | — | 4 | pruned | — |
| k=10000, seg=0.1, zero, n=48 | 12.3486 | 1.0062 | 2.1200 | — | — | 4 | pruned | — |
| k=16000, seg=0.12, hybrid rho=0.8, n=48 | 12.3440 | 1.0365 | 2.1634 | — | — | 5 | pruned | — |
| k=16000, seg=0.1, zero, n=48 | 12.3158 | 1.0200 | 2.1372 | — | — | 4 | pruned | — |
| k=6000, seg=0.12, zero, n=48 | 12.3114 | 1.0280 | 2.1535 | — | — | 5 | pruned | — |
| k=32000, seg=0.1, hybrid rho=0.8, n=48 | 12.3017 | 1.0078 | 2.1118 | — | — | 4 | pruned | — |
| k=10000, seg=0.1, hybrid rho=0.8, n=48 | 12.2981 | 1.0145 | 2.1307 | — | — | 4 | pruned | — |
| k=8000, seg=0.12, zero, n=48 | 12.2971 | 1.0751 | 2.2118 | — | — | 5 | pruned | — |
| k=32000, seg=0.1, zero, n=48 | 12.2940 | 1.0136 | 2.1191 | — | — | 4 | pruned | — |
| k=8000, seg=0.12, hybrid rho=0.8, n=48 | 12.2903 | 1.0682 | 2.1963 | — | — | 5 | pruned | — |
| k=6000, seg=0.1, hybrid rho=0.8, n=48 | 12.2615 | 1.0142 | 2.1373 | — | — | 4 | pruned | — |
| k=16000, seg=0.1, hybrid rho=0.8, n=48 | 12.2596 | 1.0128 | 2.1321 | — | — | 4 | pruned | — |
| k=6000, seg=0.1, zero, n=48 | 12.2403 | 1.0075 | 2.1162 | — | — | 4 | pruned | — |
| k=8000, seg=0.1, zero, n=48 | 12.1733 | 1.0345 | 2.1408 | — | — | 4 | pruned | — |
| k=8000, seg=0.1, hybrid rho=0.8, n=48 | 12.1545 | 1.0411 | 2.1619 | — | — | 4 | pruned | — |

## Stage B

| Method | AOC | Runtime (s) | Benchmark (s) | delta vs IG | delta vs NAA | seg steps | Status | Reason |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| k=32000, seg=0.2, zero, n=48 | 12.7864 | 1.1147 | 2.2719 | — | — | 9 | selected | carried_zero |
| k=4000, seg=0.15, hybrid rho=1, n=48 | 12.6795 | 0.9958 | 2.1301 | — | — | 7 | selected | — |
| k=4000, seg=0.15, zero, n=48 | 12.6733 | 1.0540 | 2.1962 | — | — | 7 | selected | carried_zero |
| k=4000, seg=0.15, hybrid rho=0.6, n=48 | 12.6264 | 0.9976 | 2.1239 | — | — | 7 | selected | — |
| k=4000, seg=0.15, hybrid rho=0.8, n=48 | 12.6112 | 1.0494 | 2.2038 | — | — | 7 | selected | carried_rho0.8 |
| k=4000, seg=0.15, hybrid rho=1.2, n=48 | 12.5729 | 0.9946 | 2.1263 | — | — | 7 | selected | — |
| k=32000, seg=0.12, zero, n=48 | 12.4535 | 1.0096 | 2.1277 | — | — | 5 | pruned | carried_zero |
| k=10000, seg=0.12, zero, n=48 | 12.4464 | 1.0029 | 2.1183 | — | — | 5 | pruned | carried_zero |
| k=4000, seg=0.1, hybrid rho=1, n=48 | 12.4385 | 0.9798 | 2.0741 | — | — | 4 | selected | — |
| k=4000, seg=0.1, hybrid rho=1.2, n=48 | 12.4278 | 0.9354 | 2.0310 | — | — | 4 | selected | — |
| k=4000, seg=0.1, hybrid rho=0.8, n=48 | 12.4223 | 1.0324 | 2.1643 | — | — | 4 | pruned | carried_rho0.8 |
| k=4000, seg=0.1, hybrid rho=0.6, n=48 | 12.3418 | 0.9367 | 2.0271 | — | — | 4 | pruned | — |

## Stage C

| Method | AOC | Runtime (s) | Benchmark (s) | delta vs IG | delta vs NAA | seg steps | Status | Reason |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| best_quality - k=32000, seg=0.2, zero, n=24 | 12.0866 | 0.6439 | 2.9537 | — | — | 4 | finalist | — |
| k=32000, seg=0.2, zero, n=48 | 12.0866 | 1.1117 | 3.4215 | — | — | 9 | pruned | — |
| k=32000, seg=0.2, zero, n=96 | 12.0866 | 1.9751 | 4.2849 | — | — | 19 | pruned | — |
| k=32000, seg=0.2, zero, n=192 | 12.0866 | 3.6817 | 5.9915 | — | — | 38 | pruned | — |
| best_balanced - k=4000, seg=0.15, hybrid rho=0.6, n=24 | 12.0480 | 0.6244 | 2.9605 | — | — | 3 | finalist | — |
| k=4000, seg=0.15, hybrid rho=0.6, n=48 | 12.0480 | 1.0486 | 3.3847 | — | — | 7 | pruned | — |
| k=4000, seg=0.15, hybrid rho=0.6, n=96 | 12.0480 | 1.8716 | 4.2077 | — | — | 14 | pruned | — |
| k=4000, seg=0.15, hybrid rho=0.6, n=192 | 12.0480 | 3.4768 | 5.8129 | — | — | 28 | pruned | — |
| k=4000, seg=0.15, hybrid rho=1.2, n=24 | 12.0240 | 0.6237 | 2.9743 | — | — | 3 | pareto | — |
| k=4000, seg=0.15, hybrid rho=1.2, n=48 | 12.0240 | 1.0485 | 3.3991 | — | — | 7 | pruned | — |
| k=4000, seg=0.15, hybrid rho=1.2, n=96 | 12.0240 | 1.8702 | 4.2209 | — | — | 14 | pruned | — |
| k=4000, seg=0.15, hybrid rho=1.2, n=192 | 12.0240 | 3.4795 | 5.8301 | — | — | 28 | pruned | — |
| k=4000, seg=0.15, hybrid rho=1, n=24 | 12.0220 | 0.6240 | 2.9703 | — | — | 3 | pruned | — |
| k=4000, seg=0.15, hybrid rho=1, n=48 | 12.0220 | 1.0492 | 3.3956 | — | — | 7 | pruned | — |
| k=4000, seg=0.15, hybrid rho=1, n=96 | 12.0220 | 1.8699 | 4.2162 | — | — | 14 | pruned | — |
| k=4000, seg=0.15, hybrid rho=1, n=192 | 12.0220 | 3.4815 | 5.8279 | — | — | 28 | pruned | — |
| k=4000, seg=0.15, zero, n=24 | 12.0201 | 0.6228 | 2.9436 | — | — | 3 | pareto | — |
| k=4000, seg=0.15, zero, n=48 | 12.0201 | 1.0635 | 3.3844 | — | — | 7 | pruned | — |
| k=4000, seg=0.15, zero, n=96 | 12.0201 | 1.8714 | 4.1922 | — | — | 14 | pruned | — |
| k=4000, seg=0.15, zero, n=192 | 12.0201 | 3.4760 | 5.7968 | — | — | 28 | pruned | — |
| k=4000, seg=0.15, hybrid rho=0.8, n=24 | 12.0153 | 0.6247 | 2.9689 | — | — | 3 | pruned | — |
| k=4000, seg=0.15, hybrid rho=0.8, n=48 | 12.0153 | 1.0635 | 3.4077 | — | — | 7 | pruned | — |
| k=4000, seg=0.15, hybrid rho=0.8, n=96 | 12.0153 | 1.8684 | 4.2125 | — | — | 14 | pruned | — |
| k=4000, seg=0.15, hybrid rho=0.8, n=192 | 12.0153 | 3.4782 | 5.8224 | — | — | 28 | pruned | — |
| k=4000, seg=0.1, hybrid rho=1, n=24 | 11.8437 | 0.6153 | 2.9110 | — | — | 2 | pareto | — |
| k=4000, seg=0.1, hybrid rho=1, n=48 | 11.8437 | 1.0330 | 3.3287 | — | — | 4 | pruned | — |
| k=4000, seg=0.1, hybrid rho=1, n=96 | 11.8437 | 1.7654 | 4.0611 | — | — | 9 | pruned | — |
| k=4000, seg=0.1, hybrid rho=1, n=192 | 11.8437 | 3.2903 | 5.5860 | — | — | 19 | pruned | — |
| fastest_pareto - k=4000, seg=0.1, hybrid rho=1.2, n=24 | 11.8371 | 0.6029 | 2.9037 | — | — | 2 | finalist | — |
| k=4000, seg=0.1, hybrid rho=1.2, n=48 | 11.8371 | 0.9872 | 3.2879 | — | — | 4 | pruned | — |
| k=4000, seg=0.1, hybrid rho=1.2, n=96 | 11.8371 | 1.7648 | 4.0655 | — | — | 9 | pruned | — |
| k=4000, seg=0.1, hybrid rho=1.2, n=192 | 11.8371 | 3.2889 | 5.5897 | — | — | 19 | pruned | — |

## Figures

`search_candidates.csv`: `output/road_hparam_search_oxford_pets_100_pred_top1/search_candidates.csv`

### stage_a_heatmap

![](output/road_hparam_search_oxford_pets_100_pred_top1/figures/stage_a_heatmap_1776073476326309000.png)

### stage_b_rho_heatmap

![](output/road_hparam_search_oxford_pets_100_pred_top1/figures/stage_b_rho_heatmap_1776073476326309000.png)

### nsteps_tradeoff

![](output/road_hparam_search_oxford_pets_100_pred_top1/figures/nsteps_tradeoff_1776073476326309000.png)

### search_pareto_scatter

![](output/road_hparam_search_oxford_pets_100_pred_top1/figures/search_pareto_scatter_1776073476326309000.png)

### search_parameter_effects

![](output/road_hparam_search_oxford_pets_100_pred_top1/figures/search_parameter_effects_1776073476326309000.png)


# ROAD Hyperparameter Search Holdout

- layer_name=`model.6`
- holdout_images=30
- baseline_n_steps=128

## Finalists

| Label | Method | AOC | Runtime (s) | top_k | segment_end | fill | seg steps | n_steps |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| best_quality | cheap-ig[0,0.2]/positive/k32000 | 12.0866 | 0.6439 | 32000 | 0.2 | zero | 4 | 24 |
| fastest_pareto | cheap-ig[0,0.1]/positive/k4000/fill-naa_scaled-rho1.2 | 11.8371 | 0.6029 | 4000 | 0.1 | hybrid rho=1.2 | 2 | 24 |
| best_balanced | cheap-ig[0,0.15]/positive/k4000/fill-naa_scaled-rho0.6 | 12.0480 | 0.6244 | 4000 | 0.15 | hybrid rho=0.6 | 3 | 24 |

## Holdout Summary

| Method | AOC | Runtime (s) | Benchmark (s) | delta vs IG | delta vs NAA | Consistency | Abs Error |
| --- | --- | --- | --- | --- | --- | --- | --- |
| IG | 12.5100 +- 2.9330 | 4.4663 +- 0.0368 | 6.6519 +- 0.0424 | 0.0000 | 2.2150 | 0.0741 +- 0.1325 | 0.6125 +- 0.5300 |
| NAA | 10.2950 +- 3.0255 | 1.9160 +- 0.0378 | 4.0439 +- 0.0459 | -2.2150 | 0.0000 | 0.1185 +- 0.1858 | 13.6707 +- 2.9596 |
| best_quality: cheap-ig[0,0.2]/positive/k32000 | 13.1320 +- 3.1094 | 0.6367 +- 0.0057 | 2.9436 +- 0.0579 | 0.6220 | 2.8370 | 0.0593 +- 0.0940 | 122.7652 +- 37.2865 |
| fastest_pareto: cheap-ig[0,0.1]/positive/k4000/fill-naa_scaled-rho1.2 | 13.1725 +- 3.0084 | 0.5956 +- 0.0051 | 2.8869 +- 0.0645 | 0.6625 | 2.8775 | 0.0741 +- 0.1086 | 165.9039 +- 67.4444 |
| best_balanced: cheap-ig[0,0.15]/positive/k4000/fill-naa_scaled-rho0.6 | 13.2600 +- 2.9929 | 0.6192 +- 0.0041 | 2.9455 +- 0.0599 | 0.7500 | 2.9650 | 0.0704 +- 0.1053 | 132.1404 +- 45.2611 |

## Figures

### holdout_curves

![](output/road_hparam_search_oxford_pets_100_pred_top1/figures/holdout_curves_1776073476326309000.png)

### holdout_summary

![](output/road_hparam_search_oxford_pets_100_pred_top1/figures/holdout_summary_1776073476326309000.png)


## Сравнение baseline choices из литературы

Для path-attribution методов в литературе встречаются разные baseline-стратегии:

- `zero` / black image: [Sundararajan et al., 2017](https://proceedings.mlr.press/v70/sundararajan17a.html).
- random-noise / uniform baselines: обсуждаются у [Sundararajan et al., 2017](https://proceedings.mlr.press/v70/sundararajan17a.html) и в обзоре [Sturmfels et al., 2020](https://distill.pub/2020/attribution-baselines/).
- constant-color / average-color и blurred-image baselines: [Fong & Vedaldi, 2017](https://openaccess.thecvf.com/content_ICCV_2017/html/Fong_Interpretable_Explanations_of_ICCV_2017_paper.html).
- baseline distributions / expected gradients: [Erion et al., 2021](https://www.nature.com/articles/s42256-021-00343-w).
- blur-path как способ уйти от выбора single baseline: [Xu et al., 2020](https://openaccess.thecvf.com/content_CVPR_2020/html/Xu_Attribution_in_Scale_and_Space_CVPR_2020_paper.html).

Ниже для практического сравнения берём три детерминированных и недорогих варианта: `zero`, `mean_rgb` (операционализация constant-color baseline) и `blur`. Search остаётся тем же, что и раньше, а baseline сравнивается отдельно на holdout для тех же finalists, чтобы изолировать именно эффект baseline.


In [ ]:
BASELINE_COMPARISON_CASES = [
    {"key": "zero", "mode": "zero"},
    {"key": "mean_rgb", "mode": "mean_rgb", "baseline_rgb": baseline_mean_rgb},
    {"key": "blur", "mode": "blur", "baseline_blur_sigma": 16.0},
]
BASELINE_COMPARISON_OUTPUT = OUTPUT_DIR / "baseline_comparison.md"


def finalist_method_spec(finalist, baseline_case):
    config = dict(finalist["config"])
    kwargs = {
        "selection_mode": config["selection_mode"],
        "selection_top_k": int(config["selection_top_k"]),
        "segment_start": float(config["segment_start"]),
        "segment_end": float(config["segment_end"]),
        "fill_mode": config["fill_mode"],
    }
    if config["fill_mode"] == "naa_scaled" and config["fill_rho"] is not None:
        kwargs["fill_rho"] = float(config["fill_rho"])
    if baseline_case["mode"] != "zero":
        kwargs["baseline_mode"] = baseline_case["mode"]
        if baseline_case["mode"] == "mean_rgb":
            kwargs["baseline_rgb"] = [float(v) for v in baseline_case["baseline_rgb"]]
        elif baseline_case["mode"] == "blur":
            kwargs["baseline_blur_sigma"] = float(baseline_case.get("baseline_blur_sigma", 16.0))
    return classifier_method_spec("cheap_ig", **kwargs)


def run_finalist_baseline_comparison(finalists, baseline_cases, holdout_image_paths):
    grouped_specs = {}
    spec_meta = {}

    for finalist_index, finalist in enumerate(finalists, start=1):
        for case in baseline_cases:
            spec = finalist_method_spec(finalist, case)
            grouped_specs.setdefault(int(finalist["n_steps"]), []).append(spec)
            spec_meta[spec["name"]] = {
                "finalist_label": finalist["label"],
                "finalist_rank": finalist_index,
                "baseline_key": case["key"],
                "baseline_label": baseline_display_name(
                    case["mode"],
                    baseline_rgb=case.get("baseline_rgb"),
                    blur_sigma=case.get("baseline_blur_sigma"),
                ),
                "method_name": spec["name"],
                "n_steps": int(finalist["n_steps"]),
            }

    grouped_results = []
    for n_steps in sorted(grouped_specs):
        grouped_results.append(
            benchmark_classifier_road(
                image_paths=holdout_image_paths,
                method_specs=grouped_specs[n_steps],
                layer_name=CLASSIFIER_LAYER,
                n_steps=n_steps,
                percentiles=results["stages"]["holdout"]["percentiles"],
                noise=ROAD_NOISE,
                noise_seed=ROAD_NOISE_SEED,
                cache_root=CACHE_ROOT,
                save_output=False,
                fd_eps=FD_EPS,
                clear_every=CLEAR_EVERY,
                refresh_core=REFRESH_CORE,
                refresh_methods=REFRESH_METHODS,
                refresh_evaluations=REFRESH_EVALUATIONS,
                verbose=False,
            )
        )

    records = []
    by_method_name = {}
    for result in grouped_results:
        for spec in result["method_specs"]:
            stats = result["summary"]["method_summaries"][spec["name"]]
            meta = dict(spec_meta[spec["name"]])
            meta.update(
                {
                    "aoc_mean": stats["target_logit_drop_aoc"]["mean"],
                    "aoc_std": stats["target_logit_drop_aoc"]["std"],
                    "runtime_mean": stats["runtime_s"]["mean"],
                    "runtime_std": stats["runtime_s"]["std"],
                    "abs_error_mean": stats["abs_error"]["mean"],
                    "abs_error_std": stats["abs_error"]["std"],
                    "consistency_mean": stats["road_morf_mean_consistency"]["mean"],
                }
            )
            records.append(meta)
            by_method_name[spec["name"]] = meta

    zero_case = next(case for case in baseline_cases if case["key"] == "zero")
    for finalist in finalists:
        zero_name = finalist_method_spec(finalist, zero_case)["name"]
        zero_mean = by_method_name[zero_name]["aoc_mean"]
        for case in baseline_cases:
            method_name = finalist_method_spec(finalist, case)["name"]
            by_method_name[method_name]["delta_vs_zero"] = by_method_name[method_name]["aoc_mean"] - zero_mean

    case_order = {case["key"]: idx for idx, case in enumerate(baseline_cases)}
    records.sort(key=lambda record: (record["finalist_rank"], case_order[record["baseline_key"]]))
    return records


def format_stats(mean_value, std_value):
    if mean_value != mean_value:
        return "—"
    return f"{float(mean_value):.4f} +- {float(std_value):.4f}"


def build_baseline_comparison_markdown(records, baseline_cases):
    lines = [
        "# Cheap-IG baseline comparison on ROAD holdout",
        "",
        "Сравнение делается для тех же finalists, которые были выбраны staged-search при `zero` baseline. Это позволяет не смешивать эффект baseline с повторным search по гиперпараметрам.",
        "",
        f"- holdout_images={len(split['holdout_image_paths'])}",
        f"- percentiles={results['stages']['holdout']['percentiles']}",
        "- mean_rgb_baseline=({})".format(", ".join(f"{value:.4f}" for value in baseline_mean_rgb)),
        "- blur_sigma=16.0",
        "",
        "## Per-finalist table",
        "",
        "| Finalist | Baseline | ROAD target-logit AOC | Δ vs zero | Attr runtime (s) | Mean consistency | Abs error |",
        "| --- | --- | ---: | ---: | ---: | ---: | ---: |",
    ]

    for record in records:
        lines.append(
            "| {finalist_label} | {baseline_label} | {aoc} | {delta:+.4f} | {runtime} | {consistency:.4f} | {abs_error} |".format(
                finalist_label=record["finalist_label"],
                baseline_label=record["baseline_label"],
                aoc=format_stats(record["aoc_mean"], record["aoc_std"]),
                delta=float(record.get("delta_vs_zero", 0.0)),
                runtime=format_stats(record["runtime_mean"], record["runtime_std"]),
                consistency=float(record["consistency_mean"]),
                abs_error=format_stats(record["abs_error_mean"], record["abs_error_std"]),
            )
        )

    lines.extend([
        "",
        "## Baseline aggregate",
        "",
        "| Baseline | Mean AOC across finalists | Mean Δ vs zero | Mean runtime (s) |",
        "| --- | ---: | ---: | ---: |",
    ])

    for case in baseline_cases:
        case_records = [record for record in records if record["baseline_key"] == case["key"]]
        mean_aoc = float(np.mean([record["aoc_mean"] for record in case_records]))
        mean_delta = float(np.mean([record.get("delta_vs_zero", 0.0) for record in case_records]))
        mean_runtime = float(np.mean([record["runtime_mean"] for record in case_records]))
        lines.append(
            "| {} | {:.4f} | {:+.4f} | {:.4f} |".format(
                baseline_display_name(
                    case["mode"],
                    baseline_rgb=case.get("baseline_rgb"),
                    blur_sigma=case.get("baseline_blur_sigma"),
                ),
                mean_aoc,
                mean_delta,
                mean_runtime,
            )
        )

    return "\n".join(lines)


baseline_comparison_records = run_finalist_baseline_comparison(
    finalists=results["finalists"],
    baseline_cases=BASELINE_COMPARISON_CASES,
    holdout_image_paths=split["holdout_image_paths"],
)
baseline_comparison_markdown = build_baseline_comparison_markdown(
    baseline_comparison_records,
    BASELINE_COMPARISON_CASES,
)
BASELINE_COMPARISON_OUTPUT.write_text(baseline_comparison_markdown + "\n", encoding="utf-8")
display(Markdown(baseline_comparison_markdown))
